# Free-Droid (Szabi) — v3 fine-tune (egyszerűsített dataset)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

**Mi új a v2-höz képest:**
- A v2 fine-tune (gentle) a base modellhez képest visszaesett koherenciában és provokáció-kezelésben —
  az ok nem a hiperparaméter, hanem a dataset nyelvi komplexitása (a kis modell a költői mondatokból
  csak töredékeket másol, attól lesz zagyva). A javítás a nyelv egyszerűsítése, nem a recept.
- Dataset 744 → 626 (train 563 / val 63): minden válasz rövid, hétköznapi, metafora nélkül
  (egy gyerek is értse); a metafora-csapda és absztrakt-univerzális kérdések kidobva; a tudás a RAG-ba
  került; +21 egyszerű identitás-példa (freedroid-ext.json). A tool-példák (130, 100% kanonikus) érintetlenek.
- Recept változatlan: `--preset gentle` (lr 5e-5, 1 epoch, r/alpha 8) — a gentle marad, mert nem a
  hiperparaméter volt a baj.

**Először: Runtime → Change runtime type → T4 GPU.**

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása

Az egyszerűsített 626-os dataset a **`main`-en** van (PR #15 mergelve). A notebook a main-t klónozza.
(Ha mégis egy feature-branchről futtatnál, írd át a lenti `-b main` kapcsolót a branch nevére.)

In [ ]:
# 2. Repo a main-ről (a tiszta dataset + finetune.py), majd be a training/-be.
!git clone --depth 1 -b main https://github.com/pits2022/free-droid.git
%cd free-droid/training
# Gyors ellenőrzés: a dataset a 626-os, egyszerűsített legyen.
!python -c "import json; d=json.load(open('dataset/freedroid_full.json')); print('peldak:', len(d))"
!wc -l dataset/train.jsonl dataset/val.jsonl

## Edge modell — Llama 3.2 3B (offline fallback)

A Pi 5-ön, teljesen offline futó modell. Q4_K_M (edge) + Q8_0 (cloud) GGUF export.
Kimenet: `outputs/llama3.2-3b-v3/`.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v3

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A CAX31-en CPU-n futó nagyobb modell — a magyar/persona minőséget a méret hozza. A `llama8b` variáns
T4-biztos (batch=1, grad_accum=8) és csak Q4_K_M-et exportál. Kimenet: `outputs/llama3.1-8b-v3/`.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v3

## Next

- Kimenetek: `training/outputs/<variant>-v3/` — `gguf-q4_k_m` (edge), `gguf-q8_0` (cloud), `lora-adapter`.
  Töltsd le (vagy push HF Hubra) — git-ignorálva.
- **Ollama:** `ollama create szabi-3b-v3 -f Modelfile` (a `FROM` a kiválasztott GGUF-ra). Ügyelj a helyes
  `TEMPLATE`-re (Llama-3 fejléc) + stop tokenekre — az Unsloth GGUF NEM ágyazza be a chat sablont.
- **Benchmark:** `run_benchmark.py --models szabi-8b-v3 llama3.1:8b szabi-3b-v3 llama3.2:3b` — a cél, hogy
  a v3 visszahozza a base koherenciáját/provokáció-kezelését, megtartva a tool/persona-tudást.
  RAG-összevetés: `--rag`.
- **Kötelező red-team pass** (provokatív / offtopic) a demó előtt.
- A tényeket futásidőben a `freedroid.rag` (offline BM25) adja, nem a fine-tune.